# RandomForest 모델 구성 설명

## 목표와 사용 데이터

- 사용 데이터: `demo_1/used_data/weekly_next_week_with_vle_enhanced.csv`
- 데이터 단위: 학생×과목×운영회차×예측 주차
- Target: `target_next_week_withdrawn` — w주차 종료 시점 Feature로 w+1주차 이탈 여부를 예측
- 입력 Feature: `id_student`, Target을 제외한 124개 Feature
- `id_student`는 모델에 입력하지 않고, 동일 학생의 주차별 행이 학습·검증에 섞이지 않도록 GroupKFold 분할에만 사용

## 전처리와 검증

- 범주형 Feature는 `OneHotEncoder(handle_unknown='ignore')`로 변환한다.
- Encoder는 각 학습 Fold에만 적합해 검증 Fold의 범주 정보가 미리 반영되지 않게 한다.
- 수치형 Feature의 무한값은 결측값으로 바꾼다.
- 검증은 `GroupKFold(id_student, 3-Fold)`를 사용한다.

## 랜덤포레스트 설정

|항목|설정|이유|
|---|---:|---|
|`n_estimators`|300|여러 트리의 평균으로 예측 분산을 줄임|
|`max_depth`|18|매우 깊은 트리로 인한 과적합 완화|
|`min_samples_split`|20|작은 표본만으로 과도하게 분기되는 것을 제한|
|`min_samples_leaf`|10|리프 노드 예측확률의 안정성 확보|
|`max_features`|0.7|트리 간 다양성 확보|
|`max_samples`|0.8|각 트리의 학습 표본을 줄여 학습 시간·상관성 완화|
|`class_weight`|`balanced_subsample`|극소수 이탈 클래스의 손실 비중 보정|

## 평가 지표

OOF 예측확률로 Recall@Top-20%, PR-AUC, Brier Score, ECE(10-bin)를 계산한다. 랜덤포레스트는 확률이 과도하게 극단적일 수 있으므로, PR-AUC뿐 아니라 Brier Score와 ECE를 CatBoost와 함께 비교한다.

In [ ]:
import runpy

# 현재 노트북과 같은 models 폴더에 있는 학습 스크립트를 실행한다.
runpy.run_path('05_randomforest_weekly_next_week.py', run_name='__main__')

## 생성 결과 파일

- `demo_1/randomforest_weekly_next_week_metrics.csv`: 전체 OOF 성능
- `demo_1/randomforest_weekly_next_week_fold_metrics.csv`: Fold별 성능
- `demo_1/randomforest_weekly_next_week_oof_predictions.csv`: 학생·과목·주차별 OOF 예측확률

> OOF 예측확률 파일은 대용량이므로 `.gitignore` 적용 대상이다.